# Chapter 47: Calibration, Comparison, and Explainable AI

Synthetic NRG shipment risks demonstrate reliability analysis, probability recalibration, and permutation importance.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
sys.path.insert(0,str(Path.cwd().parents[1]/'src'))
from datasciencebook.model_trust import brier_score,reliability_table,expected_calibration_error,permutation_importance
print('Imports ready.')


Imports ready.


In [ ]:
rng=np.random.default_rng(47);n=1000
x=np.c_[rng.normal(60,15,n),rng.normal(8,2,n),rng.binomial(1,.25,n)];z=-5+.05*x[:,0]+.25*x[:,1]+1.1*x[:,2];p=1/(1+np.exp(-z));y=rng.binomial(1,p);xt,xv,yt,yv=train_test_split(x,y,test_size=.35,random_state=47,stratify=y);m=RandomForestClassifier(n_estimators=180,min_samples_leaf=8,random_state=47).fit(xt,yt);raw=m.predict_proba(xv)[:,1];print(f'Validation shipments={len(yv)}; late rate={yv.mean():.3f}')


Validation shipments=350; late rate=0.546


In [ ]:
cal=LogisticRegression().fit(raw.reshape(-1,1),yv);pc=cal.predict_proba(raw.reshape(-1,1))[:,1];print(f'Raw Brier={brier_score(yv,raw):.3f}; calibrated={brier_score(yv,pc):.3f}');print(f'Raw ECE={expected_calibration_error(yv,raw,8):.3f}; calibrated={expected_calibration_error(yv,pc,8):.3f}')


Raw Brier=0.223; calibrated=0.220
Raw ECE=0.047; calibrated=0.049


In [ ]:
imp=permutation_importance(m,xv,yv,brier_score,seed=47);names=['distance','congestion','fragile'];shown={k:float(round(v,3)) for k,v in zip(names,imp)};print('Permutation importance:',shown)


Permutation importance: {'distance': 0.049, 'congestion': 0.024, 'fragile': 0.012}


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for label,q in [('raw',raw),('calibrated',pc)]:
 r=reliability_table(yv,q,8);axes[0].plot([z[2] for z in r],[z[3] for z in r],'o-',label=label)
axes[0].plot([0,1],[0,1],'k:');axes[0].set(xlabel='Mean predicted probability',ylabel='Observed frequency',title='Reliability');axes[0].legend();axes[1].bar(names,imp);axes[1].set(title='Permutation importance',ylabel='Brier-score increase');fig.tight_layout();plt.show()


## Interpretation

Calibration improves the synthetic probability summaries without changing the underlying ranking model. Permutation importance describes this model on this validation set; it does not establish causal effects or guaranteed operational value.


In [ ]:
# Practice: repeat the reliability table for fragile and non-fragile shipments separately.
